# Fine-tune mSWE-GNN on Ahr river (Google Colab)

This notebook fine-tunes the pre-trained dk15 model on the Ahr river dataset.

**Before running:**
1. Upload your repo zip to Google Drive (or clone from GitHub)
2. Set `DRIVE_PATH` below to where you placed the repo
3. Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ── 1. Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 2. Set path and unzip repo ───────────────────────────────────────────────
import os

# Path to your zipped repo on Google Drive
REPO_ZIP = '/content/drive/MyDrive/mSWE-GNN_marg.zip'  # <-- change if needed
REPO_DIR = '/content/mSWE-GNN_marg'

if not os.path.exists(REPO_DIR):
    os.makedirs(REPO_DIR, exist_ok=True)
    !unzip -q "{REPO_ZIP}" -d /content/
    print('Unzipped.')
else:
    print('Repo already extracted.')

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ── 3. Install dependencies ──────────────────────────────────────────────────
# torch + torch_geometric: pin to versions compatible with Colab's CUDA
import subprocess, sys

TORCH_VERSION = '2.3.0'
CUDA_TAG = 'cu121'  # Colab T4 uses CUDA 12.x

!pip install -q torch=={TORCH_VERSION} --index-url https://download.pytorch.org/whl/{CUDA_TAG}
!pip install -q torch_geometric
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_TAG}.html
!pip install -q lightning wandb meshkernel scipy xarray

print('Dependencies installed.')

In [ ]:
# ── 4. Verify GPU ────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── 5. Run fine-tuning ───────────────────────────────────────────────────────
# Adjust --epochs and --lr as needed.
# With 1 simulation and ~20 rollout windows, 50-100 epochs is a good start.
!python finetune_ahr.py \
    --config config_finetune.yaml \
    --epochs 100 \
    --lr 5e-4 \
    --output results/finetuned_ahr.h5

In [ ]:
# ── 6. Copy fine-tuned model back to Google Drive ────────────────────────────
import shutil

SAVE_TO_DRIVE = '/content/drive/MyDrive/finetuned_ahr.h5'  # <-- change if needed
shutil.copy('results/finetuned_ahr.h5', SAVE_TO_DRIVE)
print('Saved to Drive:', SAVE_TO_DRIVE)